# 🤖 04 · Extensión con Transformers (DistilBERT) — GoEmotions
## Computación y Estructuras Discretas III (Unidad 3: Máquinas de Turing y Aplicaciones)
**Autor:** Nicholas Solís Pinilla (`NSP2007`)  
**Equipo:** JMChildrens  
**Objetivo:** Implementar, entrenar y evaluar un modelo de lenguaje preentrenado basado en Transformers (**DistilBERT**) para la clasificación binaria de sentimientos (positivo vs. negativo) sobre el corpus **GoEmotions**, comparando su rendimiento frente al *Baseline Dummy*, *Dense NN*, *Vanilla RNN* y *LSTM*, y discutiendo su equivalencia conceptual frente a las **Máquinas de Turing**.


In [ ]:
import os
import sys
import time
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    cohen_kappa_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# Configuración de rutas del proyecto
ROOT_DIR = Path.cwd().resolve().parent
DATA_DIR = ROOT_DIR / "data" / "processed"
METRICS_DIR = ROOT_DIR / "outputs" / "metrics"
FIG_DIR = ROOT_DIR / "outputs" / "figures"

FIG_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

# Fijación de semillas para reproducibilidad estricta
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

dispositivo = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✔ Entorno configurado correctamente.")
print(f"✔ Dispositivo de cómputo activo: {dispositivo}")


---
## 1. Carga y Verificación de Datos Procesados

Utilizamos exactamente las mismas particiones balanceadas generadas por el equipo en la etapa de preparación (`train.csv` con 800 muestras y `test.csv` con 200 muestras). De este modo, la comparación entre arquitecturas se mantiene estrictamente controlada y reproducible.


In [ ]:
train_path = DATA_DIR / "train.csv"
test_path = DATA_DIR / "test.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print(f"Conjunto de entrenamiento : {train_df.shape[0]} comentarios")
print(f"Conjunto de prueba (test) : {test_df.shape[0]} comentarios")
print("\nDistribución en entrenamiento:\n", train_df['label'].value_counts().rename({0: 'Negativo (0)', 1: 'Positivo (1)'}))
print("\nDistribución en prueba:\n", test_df['label'].value_counts().rename({0: 'Negativo (0)', 1: 'Positivo (1)'}))

# Mapeo descriptivo para visualización
ID2LABEL = {0: "negativo", 1: "positivo"}
LABEL2ID = {"negativo": 0, "positivo": 1}

train_df[["text", "clean", "label"]].head(6)


---
## 2. Tokenización con DistilBERT y el Algoritmo WordPiece

A diferencia de las redes neuronales densas y recurrentes entrenadas previamente (donde se dividía el texto limpio por espacios y se creaba un vocabulario estático limitado a 5,000 palabras), **DistilBERT** utiliza el algoritmo de tokenización **WordPiece**:
1. **Sub-palabras:** Si una palabra no está en el vocabulario completo, se descompone en prefijos y subfijos identificados con `##`. Esto elimina por completo el problema de palabras desconocidas (`<unk>`).
2. **Tokens Especiales:**
   * `[CLS]` (ID 101): Marcador al inicio de cada secuencia, cuyo vector final representará el resumen contextual de todo el comentario.
   * `[SEP]` (ID 102): Marcador de separación o fin de secuencia.
   * `[PAD]` (ID 0): Relleno para que todas las secuencias del lote tengan la misma dimensión.


In [ ]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tamaño del vocabulario WordPiece :", tokenizer.vocab_size)
print("Longitud máxima admitida         :", tokenizer.model_max_length, "tokens")
print("Tokens especiales                :", dict(zip(tokenizer.all_special_tokens, tokenizer.all_special_ids)))

# Demostración del funcionamiento de WordPiece
ejemplo = "I absolutely love GoEmotions but sometimes Reddit is superunbelievable!!"
tokens_ej = tokenizer.tokenize(ejemplo)
ids_ej = tokenizer.convert_tokens_to_ids(tokens_ej)

print(f"\nTexto original : {ejemplo}")
print(f"Tokens WordPiece : {tokens_ej}")
print(f"IDs numéricos    : {ids_ej}")


In [ ]:
# Análisis de longitudes en tokens para definir MAX_LEN
longitudes_train = [len(tokenizer(t)["input_ids"]) for t in train_df["text"]]

p95 = np.percentile(longitudes_train, 95)
p99 = np.percentile(longitudes_train, 99)
print(f"Longitud mínima: {min(longitudes_train)} | Mediana: {np.median(longitudes_train):.0f} | Percentil 95: {p95:.0f} | Percentil 99: {p99:.0f} | Máx: {max(longitudes_train)}")

MAX_LEN = 64
superan = sum(1 for l in longitudes_train if l > MAX_LEN)
print(f"Comentarios que superan MAX_LEN={MAX_LEN}: {superan} de {len(longitudes_train)} ({superan/len(longitudes_train)*100:.1f}%)")

plt.figure(figsize=(7, 3.2))
plt.hist(longitudes_train, bins=25, color="#2980B9", edgecolor="black", alpha=0.8)
plt.axvline(MAX_LEN, color="#C0392B", linestyle="--", linewidth=2, label=f"MAX_LEN = {MAX_LEN}")
plt.title("Distribución de longitud en tokens (WordPiece) - GoEmotions Train")
plt.xlabel("Número de tokens")
plt.ylabel("Frecuencia")
plt.legend()
plt.tight_layout()
plt.show()


---
## 3. Preparación de Datasets para Hugging Face

Estructuramos las particiones en objetos `Dataset` de Hugging Face. El tokenizador generará dos tensores fundamentales:
* **`input_ids`**: Secuencia numérica de índices en el vocabulario.
* **`attention_mask`**: Vector binario ($1$ para tokens reales, $0$ para tokens de relleno `[PAD]`) que indica a las cabezas de auto-atención qué posiciones deben atender y cuáles deben ser ignoradas.


In [ ]:
def tokenizar_funcion(lote):
    return tokenizer(
        lote["text"],
        truncation=True,
        max_length=MAX_LEN
    )

ds_train = Dataset.from_pandas(train_df[["text", "label"]])
ds_test = Dataset.from_pandas(test_df[["text", "label"]])

ds_train = ds_train.map(tokenizar_funcion, batched=True)
ds_test = ds_test.map(tokenizar_funcion, batched=True)

# DataCollator que aplica dynamic padding por lote para optimizar memoria
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("Estructura del primer ejemplo codificado:")
print({k: ds_train[0][k] for k in ["input_ids", "attention_mask", "label"]})


---
## 4. Arquitectura del Modelo Transformer (DistilBERT)

**DistilBERT** es una versión destilada de BERT que retiene el 97% de su capacidad semántica siendo un 40% más pequeña y un 60% más rápida. 
* **Cuerpo:** 6 capas de Transformer encoder, 12 cabezas de atención por capa y dimensión oculta $d = 768$.
* **Cabeza de clasificación:** Capa lineal sobre el vector `[CLS]` con regularización por Dropout y salida de dimensión 2 (logits para sentimiento positivo y negativo).


In [ ]:
modelo = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=ID2LABEL,
    label2id=LABEL2ID
)
modelo.to(dispositivo)

n_params = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
print(f"✔ Modelo '{MODEL_NAME}' cargado con éxito en {dispositivo}.")
print(f"✔ Parámetros entrenables: {n_params:,}")


---
## 5. Función de Métricas y Configuración del Entrenamiento

Para cumplir estrictamente con los requerimientos de evaluación del proyecto, calculamos las **5 métricas estándar**:
1. **Accuracy**
2. **Precision** (ponderada para la clase positiva = 1)
3. **Recall** (sensibilidad de la clase positiva)
4. **F1-Score** (media armónica)
5. **Cohen's Kappa ($\kappa$)** (acuerdo ajustado por azar)


In [ ]:
def calcular_metricas(eval_pred):
    logits, etiquetas = eval_pred
    preds = np.argmax(logits, axis=-1)
    
    acc = accuracy_score(etiquetas, preds)
    prec = precision_score(etiquetas, preds, zero_division=0)
    rec = recall_score(etiquetas, preds, zero_division=0)
    f1 = f1_score(etiquetas, preds, zero_division=0)
    kappa = cohen_kappa_score(etiquetas, preds)
    
    return {
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "kappa": kappa
    }

# Configuración de hiperparámetros de entrenamiento
training_args = TrainingArguments(
    output_dir=str(ROOT_DIR / "outputs" / "distilbert_checkpoints"),
    num_train_epochs=3,
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    weight_decay=0.01,
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="no",
    seed=SEED,
    report_to="none"
)

trainer = Trainer(
    model=modelo,
    args=training_args,
    train_dataset=ds_train,
    eval_dataset=ds_test,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=calcular_metricas
)
print("✔ Trainer configurado con éxito.")


---
## 6. Fine-Tuning y Curvas de Aprendizaje

Procedemos a entrenar el modelo sobre los 800 comentarios de `train.csv`. Monitoreamos la evolución de la función de pérdida (`loss`) y el rendimiento sobre la partición de prueba en cada época.


In [ ]:
inicio_entrenamiento = time.time()
resultado_entrenamiento = trainer.train()
tiempo_total = time.time() - inicio_entrenamiento

print(f"✔ Entrenamiento finalizado en {tiempo_total:.2f} segundos ({tiempo_total/60:.2f} minutos).")


In [ ]:
# Extracción y graficación del historial de entrenamiento
log_history = pd.DataFrame(trainer.state.log_history)

entreno = log_history.dropna(subset=["loss"])[["epoch", "loss"]]
validacion = log_history.dropna(subset=["eval_loss"])[["epoch", "eval_loss", "eval_accuracy", "eval_f1", "eval_kappa"]]

fig, axs = plt.subplots(1, 2, figsize=(11, 3.8))

axs[0].plot(entreno["epoch"], entreno["loss"], "o-", color="#C0392B", label="Entrenamiento")
axs[0].plot(validacion["epoch"], validacion["eval_loss"], "s--", color="#2980B9", label="Evaluación")
axs[0].set_title("Función de Pérdida (Cross-Entropy)")
axs[0].set_xlabel("Época")
axs[0].set_ylabel("Pérdida")
axs[0].grid(True, linestyle=":", alpha=0.6)
axs[0].legend()

axs[1].plot(validacion["epoch"], validacion["eval_accuracy"], "o-", color="#27AE60", label="Accuracy")
axs[1].plot(validacion["epoch"], validacion["eval_f1"], "^--", color="#8E44AD", label="F1-Score")
axs[1].plot(validacion["epoch"], validacion["eval_kappa"], "d-.", color="#E67E22", label="Cohen's Kappa")
axs[1].set_title("Evolución de Métricas en Test")
axs[1].set_xlabel("Época")
axs[1].set_ylabel("Valor")
axs[1].grid(True, linestyle=":", alpha=0.6)
axs[1].legend()

plt.tight_layout()
plt.savefig(FIG_DIR / "distilbert_training_curves.png", dpi=200)
plt.show()


---
## 7. Evaluación Rigurosa en Test y Matriz de Confusión

Obtenemos las predicciones del modelo fine-tuneado sobre los 200 comentarios del conjunto de prueba independiente y calculamos el reporte de clasificación completo.


In [ ]:
predicciones = trainer.predict(ds_test)
y_true = predicciones.label_ids
y_pred = np.argmax(predicciones.predictions, axis=-1)

metricas_distilbert = {
    "accuracy": round(float(accuracy_score(y_true, y_pred)), 4),
    "precision": round(float(precision_score(y_true, y_pred, zero_division=0)), 4),
    "recall": round(float(recall_score(y_true, y_pred, zero_division=0)), 4),
    "f1": round(float(f1_score(y_true, y_pred, zero_division=0)), 4),
    "kappa": round(float(cohen_kappa_score(y_true, y_pred)), 4)
}

print("=== Reporte de Clasificación (DistilBERT en Test) ===")
print(classification_report(y_true, y_pred, target_names=["Negativo", "Positivo"], digits=3))

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(4.5, 3.8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Negativo", "Positivo"])
disp.plot(cmap="Blues", values_format="d", ax=ax)
plt.title(f"Matriz de Confusión - DistilBERT (Acc: {metricas_distilbert['accuracy']:.3f})")
plt.tight_layout()
plt.savefig(FIG_DIR / "distilbert_confusion_matrix.png", dpi=200)
plt.show()


In [ ]:
# Guardado en formato estándar outputs/metrics/distilbert.json
metricas_export = {
    "model": "distilbert",
    "accuracy": metricas_distilbert["accuracy"],
    "precision": metricas_distilbert["precision"],
    "recall": metricas_distilbert["recall"],
    "f1": metricas_distilbert["f1"],
    "kappa": metricas_distilbert["kappa"],
    "n_params": int(n_params),
    "train_seconds_per_run": round(float(tiempo_total), 1),
    "confusion_matrix": cm.tolist()
}

json_path = METRICS_DIR / "distilbert.json"
json_path.write_text(json.dumps(metricas_export, indent=2), encoding="utf-8")
print(f"✔ Métricas guardadas exitosamente en {json_path}")


---
## 8. Tabla Comparativa Global y Análisis Frente a los Demás Modelos

Consolidamos las métricas de todos los modelos desarrollados por el equipo:
* **Baseline Dummy:** Estratificado por azar ($\kappa = 0$).
* **Dense NN:** Red densa completamente conectada con Bag-of-Words / embeddings promedio.
* **Vanilla RNN:** Red recurrente de Elman simple.
* **LSTM:** Long Short-Term Memory con compuertas de olvido, entrada y salida.
* **BiLSTM:** LSTM bidireccional.
* **DistilBERT:** Transformer con mecanismo de auto-atención multi-cabeza y preentrenamiento masivo.


In [ ]:
ALL_MODELS = ["baseline_dummy", "dense", "vanilla_rnn", "lstm", "bilstm", "distilbert"]
rows = []

for m in ALL_MODELS:
    fp = METRICS_DIR / f"{m}.json"
    if fp.exists():
        d = json.loads(fp.read_text(encoding="utf-8"))
        std = d.get("std", {})
        r = {"model": d["model"]}
        for k in ["accuracy", "precision", "recall", "f1", "kappa"]:
            val = d.get(k, 0.0)
            s = std.get(k)
            r[k] = f"{val:.3f}" + (f" ± {s:.3f}" if s is not None else "")
        r["params"] = f"{d.get('n_params', '-'):,}" if isinstance(d.get('n_params'), (int, float)) else d.get('n_params', '-')
        r["sec/run"] = d.get("train_seconds_per_run", "-")
        rows.append(r)

df_comparison = pd.DataFrame(rows)
df_comparison.to_csv(METRICS_DIR / "comparison.csv", index=False)
print("✔ Archivo outputs/metrics/comparison.csv actualizado.")
df_comparison


In [ ]:
# Gráfica comparativa global de todas las arquitecturas
mets = ["accuracy", "precision", "recall", "f1", "kappa"]
model_names_plot = [m for m in ALL_MODELS if (METRICS_DIR / f"{m}.json").exists()]
data_models = {n: json.loads((METRICS_DIR / f"{n}.json").read_text(encoding="utf-8")) for n in model_names_plot}

x = np.arange(len(mets))
w = 0.8 / len(model_names_plot)
colores = ["#7F8C8D", "#3498DB", "#E67E22", "#9B59B6", "#1ABC9C", "#27AE60"]

fig, ax = plt.subplots(figsize=(10, 4.2))
for i, n in enumerate(model_names_plot):
    vals = [data_models[n].get(m, 0.0) for m in mets]
    errs = [data_models[n].get("std", {}).get(m, 0.0) for m in mets]
    ax.bar(x + (i - (len(model_names_plot)-1)/2)*w, vals, w, label=n, color=colores[i % len(colores)],
           yerr=errs, capsize=2, edgecolor="black", linewidth=0.5)

ax.set_xticks(x)
ax.set_xticklabels(["Accuracy", "Precision", "Recall", "F1-Score", "Cohen's Kappa (κ)"], fontsize=10)
ax.set_ylim(0, 1.05)
ax.set_title("Comparación Integral de Modelos — Tarea Integradora 2", fontsize=12, fontweight="bold")
ax.grid(axis="y", linestyle=":", alpha=0.6)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(FIG_DIR / "all_models_metrics_comparison.png", dpi=200)
plt.show()


---
## 9. Análisis de Errores e Inferencia Interactiva

Exploramos casos específicos de `test.csv` donde el modelo falló para comprender los retos del lenguaje informal en Reddit (sarcasmo, ambigüedad emocional y expresiones coloquiales).


In [ ]:
test_con_preds = test_df.copy()
test_con_preds["pred"] = y_pred
test_con_preds["real_label"] = test_con_preds["label"].map(ID2LABEL)
test_con_preds["pred_label"] = test_con_preds["pred"].map(ID2LABEL)

errores = test_con_preds[test_con_preds["label"] != test_con_preds["pred"]]
print(f"Total de errores en test: {len(errores)} de {len(test_df)} ({len(errores)/len(test_df)*100:.1f}%)")

print("\n--- Ejemplos de discrepancia (comentarios complejos) ---")
for _, r in errores.head(5).iterrows():
    print(f"Comentario: "{r['text']}"")
    print(f"   Real: {r['real_label'].upper()}  -->  Predicho: {r['pred_label'].upper()}\n")

def predecir_sentimiento(textos):
    modelo.eval()
    enc = tokenizer(textos, padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(dispositivo)
    with torch.no_grad():
        logits = modelo(**enc).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
    
    resultados = []
    for t, p in zip(textos, probs):
        pred_id = int(np.argmax(p))
        resultados.append({
            "comentario": t,
            "P(negativo)": round(float(p[0]), 3),
            "P(positivo)": round(float(p[1]), 3),
            "predicción": ID2LABEL[pred_id]
        })
    return pd.DataFrame(resultados)

# Probando frases personalizadas en inglés
frases_prueba = [
    "I am so proud of your progress, you did amazing!",
    "This was the worst movie I have ever seen in my life, completely unwatchable.",
    "Well, that definitely went as expected... not.",
    "Thank you so much for helping me out with this project!"
]
predecir_sentimiento(frases_prueba)


---
## 10. Discusión Conceptual: Máquinas de Turing y Mecanismos de Atención

### 10.1 Limitaciones Computacionales de las Redes Recurrentes (RNN y LSTM)
Desde la perspectiva de la teoría de la computación introducida en la **Unidad 3**:
* Una **Máquina de Turing (MT)** formal posee una cinta de longitud potencialmente infinita y una función de transición que le permite leer, escribir y desplazar el cabezal bidireccionalmente:
  $$\delta: Q \times \Gamma \rightarrow Q \times \Gamma \times \{L, R\}$$
* Una red recurrente simple (RNN) modela el procesamiento secuencial como un autómata finito con memoria vectorial acotada:
  $$h_t = \tanh(W_{hh} h_{t-1} + W_{xh} x_t + b)$$
  Al procesar un comentario, el estado oculto $h_t$ se ve forzado a comprimir toda la historia previa en una dimensión fija ($d = 32$ o $64$). Si el comentario contiene 25 palabras, la información del primer token debe sobrevivir a 24 multiplicaciones matriciales sucesivas, sufriendo el fenómeno del **desvanecimiento o explosión del gradiente** (vanishing/exploding gradients).
* Las **LSTM** mitigan parcialmente esta pérdida mediante celdas de memoria aditivas ($C_t$) controladas por compuertas (forget, input, output gates), actuando como un registro protegido. Sin embargo, su complejidad de recorrido de la cinta sigue siendo estrictamente lineal $O(T)$ en tiempo secuencial, lo que restringe el paralelismo y el flujo libre de información.

### 10.2 El Mecanismo de Auto-Atención como Direccionamiento Directo por Contenido
En contraste, la arquitectura **Transformer** reemplaza la recurrencia secuencial por el mecanismo de **Auto-Atención Multi-Cabeza (Multi-Head Self-Attention)**:
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

* **Acceso Aleatorio en la Cinta:** En lugar de mover el cabezal celda a celda a lo largo de la cinta de entrada, el mecanismo de atención calcula en paralelo la similitud de cada token con todos los demás en un solo paso computacional ($O(1)$ de distancia de interacción).
* **Memoria Direccionable por Contenido:** Las matrices $Q$ (*Query*) y $K$ (*Key*) actúan como un sistema de indexación asociativa sobre la cinta: el modelo consulta *"¿qué otros tokens en la cinta son semánticamente relevantes para interpretar la palabra actual?"*. Esto permite capturar dependencias a larga distancia, negaciones lejanas e ironías que las RNN pierden.
* **Preentrenamiento y Capacidad de Generalización:** Mientras que las redes recurrentes de la Tarea 3 se entrenaron desde cero con únicamente 800 muestras de Reddit, DistilBERT llega a la tarea con un "alfabeto interno" y una comprensión sintáctica profunda adquirida al procesar miles de millones de palabras durante su preentrenamiento auto-supervisado (Masked Language Modeling). Esto explica la superioridad contundente de sus métricas de F1 y Cohen's Kappa.
